In [1]:
import os
import warnings
import random
import copy

import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

from gensim.models import Word2Vec

In [2]:
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    
    warnings.filterwarnings("ignore")
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)
    
    print(f"Random seed set to {seed}")

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2 ** 32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

In [3]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42
Device: cuda


In [4]:
INPUT_ROOT = "/kaggle/input/datasets/jun1801/df2026-data"
WORK_DIR = "/kaggle/working"

X_TRAIN = f"{INPUT_ROOT}/X_train.csv"
X_VAL = f"{INPUT_ROOT}/X_val.csv"
X_TEST = f"{INPUT_ROOT}/X_test.csv"
Y_TRAIN = f"{INPUT_ROOT}/Y_train.csv"
Y_VAL = f"{INPUT_ROOT}/Y_val.csv"

SUBMISSION_FILE = f"{WORK_DIR}/submission.csv"
CHECKPOINT_FILE = f"{WORK_DIR}/best_model.pth"

In [5]:
x_train = pd.read_csv(X_TRAIN)
x_val = pd.read_csv(X_VAL)
x_test = pd.read_csv(X_TEST)
y_train = pd.read_csv(Y_TRAIN)
y_val = pd.read_csv(Y_VAL)

In [6]:
SEQ_LENGTH = 37
FEATURE_COLS = [f"feature_{i}" for i in range(1, SEQ_LENGTH + 1)]

ATTRIBUTE_LIST = [1, 2, 3, 4, 5, 6]
ATTRIBUTE_COLS = [f"attr_{i}" for i in ATTRIBUTE_LIST]

y_all = pd.concat([y_train, y_val], ignore_index=True)
NUM_CLASSES_LIST = [(int(y_all[col].max()) + 1) for col in ATTRIBUTE_COLS]

BATCH_SIZE = 256
SHORT_SEQ_LENGTH = 5
DUPLICATE_FACTOR = 40

EMBEDDING_DIM = 256
DILATIONS = [1, 2, 4]
DROPOUT_RATE = 0.3

NUM_EPOCHS_STAGE_1 = 50
NUM_EPOCHS_STAGE_2 = 150

EARLY_STOPPING_STAGE_1 = 8
EARLY_STOPPING_STAGE_2 = 15

ALPHA_MAX_LOSS = 2.0

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-2

SCHEDULE_LR = 1e-3
PCT_START = 0.1
DIV_FACTOR = 25.0
FINAL_DIV_FACTOR = 1000.0

In [7]:
def clean_data(x_train, y_train, x_val, y_val, x_test,
               feature_cols=FEATURE_COLS,
               attr_cols=ATTRIBUTE_COLS):
    for df in [x_train, x_val, x_test]:
        df[feature_cols] = df[feature_cols].fillna(0).astype(np.int64)
        
    for df in [y_train, y_val]:
        df[attr_cols] = df[attr_cols].fillna(0).astype(np.int64)

    train_keep_idx = x_train.drop_duplicates(subset=feature_cols, keep="first").index
    x_train = x_train.loc[train_keep_idx].reset_index(drop=True)
    y_train = y_train.loc[train_keep_idx].reset_index(drop=True)

    val_keep_idx = x_val.drop_duplicates(subset=feature_cols, keep="first").index
    x_val = x_val.loc[val_keep_idx].reset_index(drop=True)
    y_val = y_val.loc[val_keep_idx].reset_index(drop=True)

    train_features_unique = x_train[feature_cols].copy()
    train_features_unique["is_in_train"] = True 
    x_val["old_idx"] = x_val.index
    val_merged = x_val.merge(train_features_unique, on=feature_cols, how="left")
    clean_val_idx = val_merged[val_merged["is_in_train"].isna()]["old_idx"]
    x_val = x_val.loc[clean_val_idx].drop(columns=["old_idx"]).reset_index(drop=True)
    y_val = y_val.loc[clean_val_idx].reset_index(drop=True)
    
    return x_train, y_train, x_val, y_val, x_test

In [8]:
x_train, y_train, x_val, y_val, x_test = clean_data(x_train, y_train, x_val, y_val, x_test)


In [9]:
seq_lengths = (x_train[FEATURE_COLS] != 0).sum(axis=1)

short_seq_mask = seq_lengths <= 5
x_short = x_train[short_seq_mask]
y_short = y_train[short_seq_mask]

num_short_seqs = len(x_short)
total_seqs = len(x_train)

DUPLICATE_FACTOR = 40

if num_short_seqs > 0:
    x_train_oversampled = pd.concat([x_train] + [x_short] * DUPLICATE_FACTOR, ignore_index=True)
    y_train_oversampled = pd.concat([y_train] + [y_short] * DUPLICATE_FACTOR, ignore_index=True)
    
    x_train = x_train_oversampled
    y_train = y_train_oversampled

In [10]:
y_all = pd.concat([y_train, y_val], ignore_index=True)
NUM_CLASSES_LIST = [(int(y_all[col].max())) for col in ATTRIBUTE_COLS]

feature_cols = [col for col in x_train.columns if col != 'id']
all_x_data = pd.concat([
    x_train[FEATURE_COLS], x_val[feature_cols], x_test[feature_cols]
    ], axis=0)

unique_ids = pd.unique(all_x_data.values.ravel())
unique_ids = unique_ids[~pd.isna(unique_ids)] 

unique_ids = [uid for uid in unique_ids if uid != 0]
id_to_index = {raw_id: idx for idx, raw_id in enumerate(sorted(unique_ids), start=1)}
id_to_index[0] = 0 

VOCAB_SIZE = max(id_to_index.values()) + 1

for df in [x_train, x_val, x_test]:
    for col in FEATURE_COLS:
        df[col] = df[col].map(id_to_index).fillna(0).astype(np.int64)

In [11]:
x_train_clean = x_train.drop(columns=['id']).values.astype(int)
x_val_clean = x_val.drop(columns=['id']).values.astype(int)
x_test_clean = x_test.drop(columns=['id']).values.astype(int)

In [12]:
all_x_data = np.vstack([x_train_clean, x_val_clean, x_test_clean]) 
max_item_id = int(np.max(all_x_data))
PAD_TOKEN = 0
MASK_TOKEN = max_item_id + 1
VOCAB_SIZE = MASK_TOKEN + 1 

print(f"Tổng số chuỗi Pre-train: {all_x_data.shape[0]} dòng")
print(f"ID mặt hàng lớn nhất: {max_item_id}")
print(f"MASK_TOKEN được gán là: {MASK_TOKEN}")
print(f"Kích thước VOCAB_SIZE truyền vào mô hình: {VOCAB_SIZE}")

Tổng số chuỗi Pre-train: 104484 dòng
ID mặt hàng lớn nhất: 254
MASK_TOKEN được gán là: 255
Kích thước VOCAB_SIZE truyền vào mô hình: 256


In [13]:
all_x_data[0]

array([  7, 173, 241,  64,   2,   7,  12,  45,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0])

In [14]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random

class MaskedBehaviorDataset(Dataset):
    def __init__(self, all_x_data, vocab_size, mask_token, pad_token=0, mask_prob=0.15):
        self.x_data = all_x_data
        self.vocab_size = vocab_size
        self.mask_token = mask_token
        self.pad_token = pad_token
        self.mask_prob = mask_prob

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        seq = self.x_data[idx].copy()
        seq_len = len(seq)
        
        masked_seq = seq.copy()
        labels = np.full(seq_len, -100) 
        
        for i in range(seq_len):
            if seq[i] == self.pad_token:
                continue 
                
            if random.random() < self.mask_prob:
                labels[i] = seq[i]
                prob = random.random()
                if prob < 0.8:
                    masked_seq[i] = self.mask_token
                elif prob < 0.9:
                    masked_seq[i] = random.randint(1, self.vocab_size - 1)

        return torch.tensor(masked_seq, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

In [15]:
import torch.nn as nn

class BERT4RecPretrainModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_heads=4, num_layers=2, max_seq_len=500, pad_token=0):
        super().__init__()
        self.pad_token = pad_token
        
        self.item_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=embed_dim * 4, 
            dropout=0.2, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.out_linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0).expand_as(x)
        
        x_emb = self.item_embedding(x) + self.pos_embedding(positions)
        
        padding_mask = (x == self.pad_token)
        
        trans_out = self.transformer(x_emb, src_key_padding_mask=padding_mask)
        logits = self.out_linear(trans_out)
        return logits

In [16]:
import torch.optim as optim

def train_pretrain_model(model, dataloader, epochs=15, lr=5e-4, device='cuda'):
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch_x, batch_y in dataloader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)

            optimizer.zero_grad()
            
            logits = model(batch_x)

            logits = logits.view(-1, model.out_linear.out_features)
            batch_y = batch_y.view(-1)

            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Pre-train Epoch [{epoch+1:02d}/{epochs}] | Loss: {avg_loss:.4f}")
    return model

In [17]:
pretrain_dataset = MaskedBehaviorDataset(
    all_x_data=all_x_data, 
    vocab_size=VOCAB_SIZE, 
    mask_token=MASK_TOKEN, 
    pad_token=PAD_TOKEN, 
    mask_prob=0.15
)

pretrain_loader = DataLoader(
    pretrain_dataset, 
    batch_size=256, 
    shuffle=True, 
    num_workers=2
)


pretrain_model = BERT4RecPretrainModel(
    vocab_size=VOCAB_SIZE, 
    embed_dim=128, 
    num_heads=4, 
    num_layers=2, 
    pad_token=PAD_TOKEN
)

trained_pretrain_model = train_pretrain_model(
    pretrain_model, 
    pretrain_loader, 
    epochs=20, 
    lr=5e-4, 
    device=DEVICE
)

torch.save(trained_pretrain_model.state_dict(), "bert4rec_pretrained.pth")
print("Đã lưu trọng số Pre-train")

Pre-train Epoch [01/20] | Loss: 3.3771
Pre-train Epoch [02/20] | Loss: 1.8817
Pre-train Epoch [03/20] | Loss: 1.5925
Pre-train Epoch [04/20] | Loss: 1.4788
Pre-train Epoch [05/20] | Loss: 1.4210
Pre-train Epoch [06/20] | Loss: 1.3854
Pre-train Epoch [07/20] | Loss: 1.3547
Pre-train Epoch [08/20] | Loss: 1.3360
Pre-train Epoch [09/20] | Loss: 1.3208
Pre-train Epoch [10/20] | Loss: 1.3051
Pre-train Epoch [11/20] | Loss: 1.2965
Pre-train Epoch [12/20] | Loss: 1.2939
Pre-train Epoch [13/20] | Loss: 1.2798
Pre-train Epoch [14/20] | Loss: 1.2783
Pre-train Epoch [15/20] | Loss: 1.2740
Pre-train Epoch [16/20] | Loss: 1.2654
Pre-train Epoch [17/20] | Loss: 1.2576
Pre-train Epoch [18/20] | Loss: 1.2478
Pre-train Epoch [19/20] | Loss: 1.2524
Pre-train Epoch [20/20] | Loss: 1.2525
Đã lưu trọng số Pre-train


In [18]:
class UserBehaviorDataset(Dataset):
    def __init__(self, x_df, y_df=None, attr_cols=ATTRIBUTE_COLS, augment=False):
        self.x_data = x_df.drop(columns=["id"]).values
        self.augment = augment 
        
        self.has_labels = y_df is not None
        if self.has_labels:
            self.y_data = y_df[attr_cols].values - 1

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        x_tensor = torch.tensor(self.x_data[idx], dtype=torch.long)
        
        if self.augment:
            if torch.rand(1).item() < 0.5:
                actions = x_tensor[x_tensor != 0]
                n_actions = len(actions)
                total_slots = len(x_tensor) 
                
                if 0 < n_actions < total_slots:
                    new_indices = torch.randperm(total_slots)[:n_actions].sort()[0]
                    
                    dilated_x = torch.zeros_like(x_tensor)
                    dilated_x[new_indices] = actions
                    
                    x_tensor = dilated_x
                
            mask_prob = torch.rand(x_tensor.shape)
            mask_drop = (mask_prob < 0.1) & (x_tensor != 0) 
            x_tensor.masked_fill_(mask_drop, 0)
        
        if self.has_labels:
            y_tensor = torch.tensor(self.y_data[idx], dtype=torch.long)
            return x_tensor, y_tensor
        
        return x_tensor

In [19]:
def create_dataloaders(x_train, y_train, x_val, y_val, x_test, 
                       feature_cols=FEATURE_COLS,
                       short_seq_length=SHORT_SEQ_LENGTH,
                       duplicate_factor=DUPLICATE_FACTOR,
                       batch_size=BATCH_SIZE,
                       num_workers=2,
                       seed_worker=seed_worker,
                       data_generator=data_generator):
    seq_lengths = (x_train[feature_cols] != 0).sum(axis=1)
    short_seq_mask = seq_lengths <= short_seq_length
    x_short = x_train[short_seq_mask]
    y_short = y_train[short_seq_mask]

    if len(x_short) > 0 and len(x_short) < 1000:
        x_train = pd.concat([x_train] + [x_short] * duplicate_factor, ignore_index=True)
        y_train = pd.concat([y_train] + [y_short] * duplicate_factor, ignore_index=True)

    train_dataset = UserBehaviorDataset(x_train, y_train, augment=True)
    val_dataset = UserBehaviorDataset(x_val, y_val)
    test_dataset = UserBehaviorDataset(x_test, None)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, worker_init_fn=seed_worker, generator=data_generator)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    
    return train_loader, val_loader, test_loader

In [20]:
train_loader_s1, val_loader_s1, test_loader_s1 = create_dataloaders(x_train, y_train, x_val, y_val, x_test)

In [21]:
class MaskedAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_out, mask):
        scores = self.attention(lstm_out)
        mask_expanded = mask.unsqueeze(-1)
        safe_min_val = torch.finfo(scores.dtype).min
        scores = scores.masked_fill(mask_expanded, safe_min_val)
        weights = F.softmax(scores, dim=1)
        context_vector = torch.sum(weights * lstm_out, dim=1)
        return context_vector, weights

In [22]:
import torch
import torch.nn as nn

class FinetuneBehaviorModel(nn.Module):
    def __init__(self, vocab_size, num_classes_list, embed_dim=128, num_heads=4, num_layers=2, max_seq_len=500, pad_token=0):
        super().__init__()
        self.pad_token = pad_token
        
        self.item_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=embed_dim * 4, 
            dropout=0.2, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.attention = MaskedAttention(embed_dim)
        self.dropouts = nn.ModuleList([nn.Dropout(0.3) for _ in range(5)])
        hidden_dim = embed_dim // 2 
        
        self.towers = nn.ModuleList()
        for num_classes in num_classes_list:
            mlp_block = nn.Sequential(
                nn.Linear(embed_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim), 
                nn.GELU(),                 
                nn.Dropout(0.1),           
                nn.Linear(hidden_dim, num_classes)
            )
            self.towers.append(mlp_block)

    def forward(self, x):
        seq_len = x.size(1)
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0).expand_as(x)
        
        x_emb = self.item_embedding(x) + self.pos_embedding(positions)
        padding_mask = (x == self.pad_token)
        trans_out = self.transformer(x_emb, src_key_padding_mask=padding_mask)
        
        context_vector, _ = self.attention(trans_out, padding_mask)

        outputs = []
        for tower in self.towers:
            tower_out = torch.mean(
                torch.stack([tower(drop(context_vector)) for drop in self.dropouts], dim=0), 
                dim=0
            )
            outputs.append(tower_out)
            
        return tuple(outputs)

In [23]:
import numpy as np
import torch
from sklearn.utils.class_weight import compute_class_weight

def get_weighted_loss_functions(y_train_df, 
                                attr_cols=ATTRIBUTE_COLS,
                                num_classes_list=NUM_CLASSES_LIST, 
                                device=DEVICE):
    loss_fns = []
    for i, col in enumerate(attr_cols):
        y_true = y_train_df[col].dropna().values.astype(int) - 1
        
        classes = np.unique(y_true)
        raw_weights = compute_class_weight("balanced", classes=classes, y=y_true)
        
        smoothed_weights = np.sqrt(raw_weights)
        clipped_weights = np.clip(smoothed_weights, a_min=None, a_max=10.0)
        
        weight_tensor = np.ones(num_classes_list[i], dtype=np.float32)
        for cls, weight in zip(classes, clipped_weights):
            weight_tensor[cls] = weight 
            
        weight_tensor = torch.tensor(weight_tensor, dtype=torch.float32).to(device)
        criterion = torch.nn.CrossEntropyLoss(
            weight=weight_tensor, 
            label_smoothing=0.01, 
            reduction="none" 
        )
        
        loss_fns.append(criterion)
        
    return loss_fns

In [24]:
import copy
from torch.cuda.amp import GradScaler, autocast
import torch

def train_model_stage_1(model, attr_idx, train_loader, val_loader, loss_fn, optimizer, scheduler, 
                        num_epochs=NUM_EPOCHS_STAGE_1, 
                        early_stopping=EARLY_STOPPING_STAGE_1,
                        device=DEVICE):
    model = model.to(device)
    best_acc = 0.0
    best_weights = copy.deepcopy(model.state_dict())
    scaler = GradScaler()

    early_stopping_count = 0
    for epoch in range(num_epochs):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            target_y = batch_y[:, attr_idx]
                
            optimizer.zero_grad(set_to_none=True)
            
            with autocast():
                outputs = model(batch_x)
                logits = outputs[0] if isinstance(outputs, (list, tuple)) else outputs
                loss = loss_fn(logits, target_y).mean()
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            preds = torch.argmax(logits.detach(), dim=1)
            train_correct += (preds == target_y).sum().item()
            train_total += target_y.size(0)

        scheduler.step()
            
        train_acc = train_correct / train_total
            
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                
                target_y = batch_y[:, attr_idx]
                    
                with autocast():
                    outputs = model(batch_x)
                
                logits = outputs[0] if isinstance(outputs, (list, tuple)) else outputs
                
                preds = torch.argmax(logits, dim=1)
                val_correct += (preds == target_y).sum().item()
                val_total += target_y.size(0)
                
        val_acc = val_correct / val_total
        
        print(f"Attribute {ATTRIBUTE_LIST[attr_idx]} Training | "
              f"Epoch {epoch + 1:02d}/{num_epochs} | "
              f"Train/Val Acc: {train_acc:.4f} / {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
            early_stopping_count = 0
        else:
            early_stopping_count += 1

        if best_acc == 1.0 or early_stopping_count == early_stopping:
            break
            
    model.load_state_dict(best_weights)
    print(f"   => Hoàn thành Attribute {ATTRIBUTE_LIST[attr_idx]} | Best Val Acc: {best_acc:.4f} \n")
    return model

In [25]:
def train_model_stage_2(model, train_loader, val_loader, loss_fns, optimizer, scheduler, 
                        attr_lst=ATTRIBUTE_LIST,
                        num_epochs=NUM_EPOCHS_STAGE_2, 
                        alpha_max_loss=ALPHA_MAX_LOSS, 
                        early_stopping=EARLY_STOPPING_STAGE_2,
                        checkpoint_file=CHECKPOINT_FILE,
                        device=DEVICE):
    model = model.to(device)
    best_exact_match = 0.0
    scaler = GradScaler()
    
    early_stopping_count = 0
    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        
        train_exact_match_correct = 0
        train_total_samples = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad(set_to_none=True)
            with autocast():
                outputs = model(batch_x)
                
                attr_losses = []
                for i in range(len(attr_lst)):
                    attr_losses.append(loss_fns[i](outputs[i], batch_y[:, i]))
                
                stacked_losses = torch.stack(attr_losses, dim=1) 
                base_loss = torch.sum(stacked_losses, dim=1)     
                worst_loss, _ = torch.max(stacked_losses, dim=1) 
                
                sample_losses = base_loss + (alpha_max_loss * worst_loss)
                loss = torch.mean(sample_losses)
            
            scaler.scale(loss).backward()
            
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            
            preds = [torch.argmax(out.detach(), dim=1) for out in outputs]
            preds_matrix = torch.stack(preds, dim=1)
            correct_rows = torch.all(preds_matrix == batch_y, dim=1)
            
            train_exact_match_correct += correct_rows.sum().item()
            train_total_samples += batch_y.size(0)

            scheduler.step()
            
        avg_train_loss = train_loss / len(train_loader)
        train_exact_match = train_exact_match_correct / train_total_samples
        
        model.eval()
        val_loss = 0.0
        val_exact_match_correct = 0
        val_total_samples = 0
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                
                with autocast():
                    outputs = model(batch_x)
                    
                    attr_losses = []
                    for i in range(len(attr_lst)):
                        attr_losses.append(loss_fns[i](outputs[i], batch_y[:, i]))
                    
                    stacked_losses = torch.stack(attr_losses, dim=1)
                    base_loss = torch.sum(stacked_losses, dim=1)
                    worst_loss, _ = torch.max(stacked_losses, dim=1)
                    
                    sample_losses = base_loss + (alpha_max_loss * worst_loss)
                    batch_loss = torch.mean(sample_losses)
                    
                val_loss += batch_loss.item()
                
                preds = [torch.argmax(out, dim=1) for out in outputs]
                preds_matrix = torch.stack(preds, dim=1)
                correct_rows = torch.all(preds_matrix == batch_y, dim=1)
                val_exact_match_correct += correct_rows.sum().item()
                val_total_samples += batch_y.size(0)
                
        avg_val_loss = val_loss / len(val_loader)
        val_exact_match = val_exact_match_correct / val_total_samples
        
        current_lr = optimizer.param_groups[0]["lr"]
        
        print(f"Epoch {epoch + 1:02d}/{num_epochs} | "
              f"LR: {current_lr:.6f} | "
              f"Train/Val Loss: {avg_train_loss:.4f}/{avg_val_loss:.4f} | "
              f"Train/Val EM: {train_exact_match:.4f} / {val_exact_match:.4f}")
        
        if val_exact_match > best_exact_match:
            best_exact_match = val_exact_match
            torch.save(model.state_dict(), checkpoint_file)
            print(f"  => Mô hình tốt nhất! (EM: {best_exact_match:.4f})")
            early_stopping_count = 0
        else:
            early_stopping_count += 1
            print(f"  => Mô hình không cải thiện ({early_stopping_count}/{early_stopping})")

        if best_exact_match == 1.0 or early_stopping_count == early_stopping:
            print("  => Dừng mô hình")
            break

In [26]:
def prepare_model(num_classes_list, num_epochs, steps_per_epoch, device=DEVICE):
    model = FinetuneBehaviorModel(
        vocab_size=VOCAB_SIZE,
        num_classes_list=num_classes_list,
        embed_dim=128,
        num_heads=4,
        num_layers=2,
        pad_token=PAD_TOKEN
    ).to(device)
    pretrained_weights = torch.load("bert4rec_pretrained.pth", map_location=DEVICE)
    model.load_state_dict(pretrained_weights, strict=False)
    optimizer = optim.AdamW(
        model.parameters(), 
        lr=LEARNING_RATE, 
        weight_decay=WEIGHT_DECAY
    ) 
    
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, 
        max_lr=SCHEDULE_LR, 
        epochs=num_epochs, 
        steps_per_epoch=steps_per_epoch, 
        pct_start=PCT_START, 
        anneal_strategy="cos", 
        div_factor=DIV_FACTOR, 
        final_div_factor=FINAL_DIV_FACTOR
    )
    
    return model, optimizer, scheduler

In [63]:
def run_inference(model, loader, attr_lst=ATTRIBUTE_LIST, device=DEVICE):
    model.eval()
    all_predictions = {f"attr_{i}": [] for i in attr_lst}
    
    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, (list, tuple)) and len(batch) == 2:
                batch_x = batch[0]
            else:
                batch_x = batch
                
            batch_x = batch_x.to(device)
            outputs = model(batch_x)
            
            for i, j in enumerate(attr_lst):
                preds = torch.argmax(outputs[i], dim=1).cpu().numpy() + 1
                all_predictions[f"attr_{j}"].extend(preds)
                
    return all_predictions

def evaluate(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = y_true.astype(np.int64)
    y_pred = np.round(y_pred).astype(np.int64)

    exact_matches = np.all(y_true == y_pred, axis=1)
    
    accuracy = np.mean(exact_matches)
    return accuracy

In [28]:
y_test_pseudo = pd.DataFrame({"id": x_test["id"]})

loss_fns_s1 = get_weighted_loss_functions(y_train)
for attr_idx in range(6):
    model_s1, optimizer_s1, scheduler_s1 = prepare_model(
        num_classes_list=[NUM_CLASSES_LIST[attr_idx]],
        num_epochs=NUM_EPOCHS_STAGE_1,
        steps_per_epoch=len(train_loader_s1)
    )
    
    model_s1 = train_model_stage_1(
        model_s1, 
        attr_idx, 
        train_loader_s1, 
        val_loader_s1, 
        loss_fns_s1[attr_idx], 
        optimizer_s1, 
        scheduler_s1
    )
    
    model_s1.eval()
    preds_s1 = []
    with torch.no_grad():
        for batch_x in test_loader_s1:
            batch_x = batch_x.to(DEVICE)
            with autocast():
                out = model_s1(batch_x)
            preds_s1.extend(torch.argmax(out[0], dim=1).cpu().numpy())
            
    y_test_pseudo[f"attr_{ATTRIBUTE_LIST[attr_idx]}"] = preds_s1

Attribute 1 Training | Epoch 01/50 | Train/Val Acc: 0.3512 / 0.4991
Attribute 1 Training | Epoch 02/50 | Train/Val Acc: 0.7105 / 0.6754
Attribute 1 Training | Epoch 03/50 | Train/Val Acc: 0.8612 / 0.8398
Attribute 1 Training | Epoch 04/50 | Train/Val Acc: 0.9200 / 0.9618
Attribute 1 Training | Epoch 05/50 | Train/Val Acc: 0.9412 / 0.9761
Attribute 1 Training | Epoch 06/50 | Train/Val Acc: 0.9502 / 0.9826
Attribute 1 Training | Epoch 07/50 | Train/Val Acc: 0.9527 / 0.9900
Attribute 1 Training | Epoch 08/50 | Train/Val Acc: 0.9550 / 0.9925
Attribute 1 Training | Epoch 09/50 | Train/Val Acc: 0.9574 / 0.9932
Attribute 1 Training | Epoch 10/50 | Train/Val Acc: 0.9584 / 0.9937
Attribute 1 Training | Epoch 11/50 | Train/Val Acc: 0.9593 / 0.9957
Attribute 1 Training | Epoch 12/50 | Train/Val Acc: 0.9607 / 0.9967
Attribute 1 Training | Epoch 13/50 | Train/Val Acc: 0.9625 / 0.9978
Attribute 1 Training | Epoch 14/50 | Train/Val Acc: 0.9631 / 0.9979
Attribute 1 Training | Epoch 15/50 | Train/Val A

In [29]:
NUM_CLASSES_LIST

[12, 31, 99, 12, 31, 99]

In [34]:
y_test_pseudo[ATTRIBUTE_COLS] = y_test_pseudo[ATTRIBUTE_COLS] + 1

In [35]:
y_test_pseudo

,id,attr_1,attr_2,attr_3,attr_4,attr_5,attr_6
0,gpbfd,11,1,84,12,1,96
1,w22ee,9,19,55,3,12,63
2,wyw95,1,1,98,3,1,53
3,izx4w,5,1,72,1,1,77
4,c6o2d,11,1,22,12,1,1
...,...,...,...,...,...,...,...
37995,lfgzf,7,1,8,6,1,95
37996,cs1tz,7,9,36,6,16,19
37997,mw26u,6,7,96,7,5,3
37998,gqnbh,10,1,47,6,1,13


In [47]:
x_train_stage2 = pd.concat([x_train, x_test], ignore_index=True)

y_train_stage2 = pd.concat([y_train, y_test_pseudo], ignore_index=True)
y_train_stage2[ATTRIBUTE_COLS] = y_train_stage2[ATTRIBUTE_COLS].astype(np.int64)

In [50]:
train_loader_s2, val_loader_s2, _ = create_dataloaders(x_train_stage2, y_train_stage2, x_val, y_val, x_test)

In [55]:
loss_fns_s2 = get_weighted_loss_functions(y_train_stage2)

model_s2, optimizer_s2, scheduler_s2 = prepare_model(
    num_classes_list=NUM_CLASSES_LIST,
    num_epochs=NUM_EPOCHS_STAGE_2,
    steps_per_epoch=len(train_loader_s2)
)

train_model_stage_2(
    model=model_s2, 
    train_loader=train_loader_s2, 
    val_loader=val_loader_s2, 
    loss_fns=loss_fns_s2, 
    optimizer=optimizer_s2, 
    scheduler=scheduler_s2
)

Epoch 01/150 | LR: 0.000050 | Train/Val Loss: 29.0380/28.2633 | Train/Val EM: 0.0000 / 0.0000
  => Mô hình không cải thiện (1/15)
Epoch 02/150 | LR: 0.000082 | Train/Val Loss: 27.1904/26.6045 | Train/Val EM: 0.0001 / 0.0001
  => Mô hình tốt nhất! (EM: 0.0001)
Epoch 03/150 | LR: 0.000132 | Train/Val Loss: 25.7174/24.9002 | Train/Val EM: 0.0013 / 0.0010
  => Mô hình tốt nhất! (EM: 0.0010)
Epoch 04/150 | LR: 0.000199 | Train/Val Loss: 23.8118/21.8260 | Train/Val EM: 0.0080 / 0.0132
  => Mô hình tốt nhất! (EM: 0.0132)
Epoch 05/150 | LR: 0.000280 | Train/Val Loss: 21.1526/17.7171 | Train/Val EM: 0.0252 / 0.0559
  => Mô hình tốt nhất! (EM: 0.0559)
Epoch 06/150 | LR: 0.000372 | Train/Val Loss: 17.7147/11.9958 | Train/Val EM: 0.1146 / 0.4293
  => Mô hình tốt nhất! (EM: 0.4293)
Epoch 07/150 | LR: 0.000470 | Train/Val Loss: 13.3006/6.4357 | Train/Val EM: 0.3237 / 0.7122
  => Mô hình tốt nhất! (EM: 0.7122)
Epoch 08/150 | LR: 0.000570 | Train/Val Loss: 10.0399/3.5849 | Train/Val EM: 0.4428 / 0.829

In [57]:
final_model = FinetuneBehaviorModel(
    vocab_size=VOCAB_SIZE, 
    num_classes_list=NUM_CLASSES_LIST,
    embed_dim=128
)
#pretrained_weights = torch.load("bert4rec_pretrained.pth", map_location=DEVICE)
#final_model.load_state_dict(pretrained_weights, strict=False)
final_model.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=DEVICE))
final_model = final_model.to(DEVICE)

array([[ 4,  0, 58, 11,  0, 41],
       [11, 18,  4, 11, 30, 14],
       [11, 22, 55,  4,  5, 34],
       ...,
       [ 6,  0, 81,  1,  0, 18],
       [ 3,  5, 69,  9, 19,  6],
       [ 5, 23, 76,  9,  6, 49]])

In [64]:
val_predictions = run_inference(final_model, val_loader_s2)
y_pred_val = np.column_stack([val_predictions[f"attr_{i}"] for i in ATTRIBUTE_LIST])

y_true_val = y_val[ATTRIBUTE_COLS].values

macro_f1_scores = []

for i, j in enumerate(ATTRIBUTE_LIST):
    f1_i = f1_score(y_true_val[:, i], y_pred_val[:, i], average="macro")
    macro_f1_scores.append(f1_i)
    print(f"Attribute {j} Val Macro F1: {f1_i:.4f}")

exact_match_acc = evaluate(y_true_val, y_pred_val)
print(f"Exact-Match Val Accuracy: {exact_match_acc:.4f}")

Attribute 1 Val Macro F1: 0.9950
Attribute 2 Val Macro F1: 0.9983
Attribute 3 Val Macro F1: 0.9987
Attribute 4 Val Macro F1: 0.9922
Attribute 5 Val Macro F1: 0.9987
Attribute 6 Val Macro F1: 0.9989
Exact-Match Val Accuracy: 0.9962


In [65]:
test_predictions = run_inference(final_model, test_loader_s1)

submission_df = pd.DataFrame()
submission_df["id"] = x_test["id"]

for i in ATTRIBUTE_LIST:
    col_name = f"attr_{i}"
    submission_df[col_name] = np.array(test_predictions[col_name], dtype=np.uint16)

submission_df.to_csv(SUBMISSION_FILE, index=False)

print(f"File nộp bài được lưu tại: {SUBMISSION_FILE}")
print(submission_df)
print(submission_df.dtypes)

File nộp bài được lưu tại: /kaggle/working/submission.csv
          id  attr_1  attr_2  attr_3  attr_4  attr_5  attr_6
0      gpbfd      11       1      84      12       1      96
1      w22ee       9      19      55       3      12      63
2      wyw95       1       1      98       3       1      53
3      izx4w       5       1      72       1       1      77
4      c6o2d      11       1      22      12       1       1
...      ...     ...     ...     ...     ...     ...     ...
37995  lfgzf       7       1       8       6       1      95
37996  cs1tz       7       9      36       6      16      19
37997  mw26u       6       7      96       7       5       3
37998  gqnbh      10       1      47       6       1      13
37999  y16g3       4      17       9       7       7      46

[38000 rows x 7 columns]
id        object
attr_1    uint16
attr_2    uint16
attr_3    uint16
attr_4    uint16
attr_5    uint16
attr_6    uint16
dtype: object
